# Verify FDG Build + CPU/GPU Integrity

This notebook performs a fast verification flow for `mesh_to_flexible_dual_grid`:

1. Choose whether to JIT compile the current local sources or directly import an already-installed `o_voxel` package.
2. Generate Gaussian point cloud on GPU and random triangle indices on GPU.
3. Run CPU path and GPU path through the selected extension module.
4. Validate output dtypes, shapes, and CPU/GPU consistency.

No local mesh file is loaded.


In [32]:
import os
import sys
import time
import types
import importlib
import importlib.util
from pathlib import Path

import torch
from torch.utils.cpp_extension import load


In [33]:
ROOT = Path(r'/mnt/nvmefs/Projects/Part Generation/TRELLIS.2-o-voxel-gpu-mod/o-voxel').resolve()
USE_JIT = False
INSTALLED_IMPORT_NAME = 'o_voxel'

print('ROOT =', ROOT)
print('USE_JIT =', USE_JIT)
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda device =', torch.cuda.get_device_name(0))


def build_jit_extension():
    sources = [
    'src/hash/hash.cu',
    'src/convert/flexible_dual_grid.cpp',
    'src/convert/volumetic_attr.cpp',
    'src/convert/mesh_to_flexible_dual_grid_gpu/torch_bindings.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/flexible_dual_grid_gpu.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/intersection_qef.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/voxelize_mesh_oct.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/voxel_traverse_edge_dda.cu',
    'src/serialize/api.cu',
    'src/serialize/hilbert.cu',
    'src/serialize/z_order.cu',
    'src/io/svo.cpp',
    'src/io/filter_parent.cpp',
    'src/io/filter_neighbor.cpp',
    'src/rasterize/rasterize.cu',
    'src/ext.cpp',
]
    full_sources = [str(ROOT / s) for s in sources]
    missing = [s for s in full_sources if not Path(s).exists()]
    if missing:
        raise FileNotFoundError(f'Missing sources: {missing}')

    build_dir = ROOT / '.verify_build'
    build_dir.mkdir(parents=True, exist_ok=True)

    unique_suffix = f"{os.getpid()}_{time.time_ns()}_{os.urandom(4).hex()}"
    mod_name = f"o_voxel_verify_{unique_suffix}"

    max_jobs = max(1, os.cpu_count() or 1)
    os.environ['MAX_JOBS'] = str(max_jobs)
    print('MAX_JOBS =', os.environ['MAX_JOBS'])
    print('JIT module name =', mod_name)

    ext_mod = load(
        name=mod_name,
        sources=full_sources,
        extra_include_paths=[str(ROOT / 'third_party/eigen')],
        extra_cflags=['-O3', '-std=c++17'],
        extra_cuda_cflags=['-O3', '-std=c++17', '--expt-relaxed-constexpr'],
        with_cuda=True,
        build_directory=str(build_dir),
        verbose=True,
    )
    print('JIT build/link: OK')
    print('jit module path =', ext_mod.__file__)
    return ext_mod


def load_local_flexible_dual_grid(ext_mod):
    pkg = types.ModuleType('o_voxel')
    pkg.__path__ = [str(ROOT / 'o_voxel')]
    pkg._C = ext_mod
    sys.modules['o_voxel'] = pkg
    sys.modules['o_voxel._C'] = ext_mod

    convert_pkg = types.ModuleType('o_voxel.convert')
    convert_pkg.__path__ = [str(ROOT / 'o_voxel' / 'convert')]
    sys.modules['o_voxel.convert'] = convert_pkg

    spec = importlib.util.spec_from_file_location(
        'o_voxel.convert.flexible_dual_grid',
        ROOT / 'o_voxel' / 'convert' / 'flexible_dual_grid.py',
    )
    mod = importlib.util.module_from_spec(spec)
    sys.modules['o_voxel.convert.flexible_dual_grid'] = mod
    spec.loader.exec_module(mod)
    return mod


if USE_JIT:
    ext_mod = build_jit_extension()
    fdg_api = load_local_flexible_dual_grid(ext_mod)
    api_mode = 'jit'
else:
    installed_pkg = importlib.import_module(INSTALLED_IMPORT_NAME)
    ext_mod = installed_pkg._C
    fdg_api = importlib.import_module(f'{INSTALLED_IMPORT_NAME}.convert.flexible_dual_grid')
    api_mode = 'installed'
    print('installed package path =', getattr(installed_pkg, '__file__', '<package>'))
    print('installed extension path =', getattr(ext_mod, '__file__', '<extension>'))
    print('installed API path =', getattr(fdg_api, '__file__', '<module>'))

print('api_mode =', api_mode)
print('ext_mod =', ext_mod)
print('fdg_api =', fdg_api)


ROOT = /mnt/nvmefs/Projects/Part Generation/TRELLIS.2-o-voxel-gpu-mod/o-voxel
USE_JIT = False
torch = 2.6.0+cu124
cuda available = True
cuda device = NVIDIA GeForce RTX 4090
installed package path = /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/__init__.py
installed extension path = /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/_C.cpython-310-x86_64-linux-gnu.so
installed API path = /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/convert/flexible_dual_grid.py
api_mode = installed
ext_mod = <module 'o_voxel._C' from '/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/_C.cpython-310-x86_64-linux-gnu.so'>
fdg_api = <module 'o_voxel.convert.flexible_dual_grid' from '/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/convert/flexible_dual_grid.py'>


In [34]:
print('has mesh_to_flexible_dual_grid_cpu =', hasattr(ext_mod, 'mesh_to_flexible_dual_grid_cpu'))
print('has mesh_to_flexible_dual_grid_gpu =', hasattr(ext_mod, 'mesh_to_flexible_dual_grid_gpu'))
print('has intersection_occ =', hasattr(fdg_api, 'intersection_occ'))
print('has intersect_qef =', hasattr(fdg_api, 'intersect_qef'))
print('has voxelize_mesh_gpu =', hasattr(fdg_api, 'voxelize_mesh_gpu'))
print('has voxelize_edge_gpu =', hasattr(fdg_api, 'voxelize_edge_gpu'))
print('has face_qef =', hasattr(fdg_api, 'face_qef'))
print('has voxel_traverse_edge_dda_gpu =', hasattr(fdg_api, 'voxel_traverse_edge_dda_gpu'))
print('has boundary_qef =', hasattr(fdg_api, 'boundary_qef'))


has mesh_to_flexible_dual_grid_cpu = True
has mesh_to_flexible_dual_grid_gpu = True
has intersection_occ = True
has intersect_qef = True
has voxelize_mesh_gpu = True
has voxelize_edge_gpu = True
has face_qef = True
has voxel_traverse_edge_dda_gpu = True
has boundary_qef = True


In [35]:
# Synthetic geometry: Gaussian points + random triangles built on GPU.
assert torch.cuda.is_available(), 'CUDA is required for this notebook workflow.'
device = torch.device('cuda:0')

N_VERT = 12000
N_FACE = 30000
GRID = 128

assert N_VERT >= 3, 'Need at least 3 vertices to build non-degenerate triangles.'

torch.manual_seed(7)
vertices_gpu = (torch.randn(N_VERT, 3, device=device, dtype=torch.float32) * 0.20).contiguous()

# # Construct faces with guaranteed distinct vertex ids per triangle.
# v0 = torch.randint(0, N_VERT, (N_FACE,), device=device, dtype=torch.int64)
# o1 = torch.randint(1, N_VERT, (N_FACE,), device=device, dtype=torch.int64)
# v1 = (v0 + o1) % N_VERT  # guaranteed v1 != v0

# # Draw an index in [0, N_VERT - 2), then map it to [0, N_VERT) excluding {v0, v1}.
# lo = torch.minimum(v0, v1)
# hi = torch.maximum(v0, v1)
# u = torch.randint(0, N_VERT - 2, (N_FACE,), device=device, dtype=torch.int64)
# v2 = u + (u >= lo).to(torch.int64)
# v2 = v2 + (v2 >= hi).to(torch.int64)

# Construct faces with guaranteed distinct vertex ids per triangle.
v0 = torch.randint(0, N_VERT, (N_FACE,), device=device, dtype=torch.int64)
v1 = torch.randint(0, N_VERT, (N_FACE,), device=device, dtype=torch.int64)
v2 = torch.randint(0, N_VERT, (N_FACE,), device=device, dtype=torch.int64)


faces_gpu = torch.stack([v0, v1, v2], dim=1)
# assert torch.all(faces_gpu[:, 0] != faces_gpu[:, 1])
# assert torch.all(faces_gpu[:, 0] != faces_gpu[:, 2])
# assert torch.all(faces_gpu[:, 1] != faces_gpu[:, 2])

faces_gpu = faces_gpu.to(torch.int32).contiguous()
aabb = torch.tensor([[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]], device=device, dtype=torch.float32)
voxel_size = ((aabb[1] - aabb[0]) / GRID).to(torch.float32).contiguous()
grid_range = torch.tensor([[0, 0, 0], [GRID, GRID, GRID]], device=device, dtype=torch.int32)
vertices_local = (vertices_gpu - aabb[0].view(1, 3)).contiguous()

print('vertices_gpu  :', vertices_gpu.shape, vertices_gpu.dtype, vertices_gpu.device)
print('faces_gpu     :', faces_gpu.shape, faces_gpu.dtype, faces_gpu.device)
print('voxel_size    :', voxel_size, voxel_size.dtype, voxel_size.device)
print('grid_range    :', grid_range.shape, grid_range.dtype, grid_range.device)
print('vertices_local:', vertices_local.shape, vertices_local.dtype, vertices_local.device)

vertices_gpu  : torch.Size([12000, 3]) torch.float32 cuda:0
faces_gpu     : torch.Size([30000, 3]) torch.int32 cuda:0
voxel_size    : tensor([0.0078, 0.0078, 0.0078], device='cuda:0') torch.float32 cuda:0
grid_range    : torch.Size([2, 3]) torch.int32 cuda:0
vertices_local: torch.Size([12000, 3]) torch.float32 cuda:0


In [36]:
# CPU verification (through JIT module API directly)
cpu_out = ext_mod.mesh_to_flexible_dual_grid_cpu(
    vertices_local.cpu(),
    faces_gpu.cpu(),
    voxel_size.cpu(),
    grid_range.cpu(),
    1.0,
    0.2,
    1e-2,
    False,
)
cpu_ok = True

cpu_ok

True

In [37]:
# GPU verification (through JIT module API directly)
gpu_out = ext_mod.mesh_to_flexible_dual_grid_gpu(
    vertices_local,
    faces_gpu,
    voxel_size,
    grid_range,
    1.0,
    0.2,
    1e-2,
    262144,
    1024,
)
torch.cuda.synchronize(device)
gpu_ok = True

gpu_ok

True

In [38]:
def summarize_out(tag, out):
    coords, dual_vertices, intersected = out
    print(f'[{tag}] coords      :', tuple(coords.shape), coords.dtype, coords.device)
    print(f'[{tag}] dual_vertices:', tuple(dual_vertices.shape), dual_vertices.dtype, dual_vertices.device)
    print(f'[{tag}] intersected  :', tuple(intersected.shape), intersected.dtype, intersected.device)
    assert coords.dim() == 2 and coords.size(1) == 3
    assert dual_vertices.dim() == 2 and dual_vertices.size(1) == 3
    assert intersected.dim() == 2 and intersected.size(1) == 3

if cpu_ok:
    summarize_out('CPU', cpu_out)
if gpu_ok:
    summarize_out('GPU', gpu_out)

if cpu_ok and gpu_ok:
    c_cpu = int(cpu_out[0].shape[0])
    c_gpu = int(gpu_out[0].shape[0])
    print('voxel_count cpu/gpu =', c_cpu, c_gpu)
    if max(c_cpu, c_gpu) > 0:
        rel = abs(c_cpu - c_gpu) / max(c_cpu, c_gpu)
        print('relative voxel count gap =', rel)

[CPU] coords      : (1441050, 3) torch.int32 cpu
[CPU] dual_vertices: (1441050, 3) torch.float32 cpu
[CPU] intersected  : (1441050, 3) torch.bool cpu
[GPU] coords      : (1441050, 3) torch.int32 cuda:0
[GPU] dual_vertices: (1441050, 3) torch.float32 cuda:0
[GPU] intersected  : (1441050, 3) torch.bool cuda:0
voxel_count cpu/gpu = 1441050 1441050
relative voxel count gap = 0.0


In [39]:
assert cpu_ok and gpu_ok, 'Run CPU/GPU cells first.'

cpu_coords, cpu_dual, cpu_inter = cpu_out
gpu_coords, gpu_dual, gpu_inter = gpu_out

cpu_coords_i64 = cpu_coords.to(dtype=torch.int64, device='cpu').contiguous()
gpu_coords_i64 = gpu_coords.to(dtype=torch.int64, device='cpu').contiguous()
cpu_dual_f32 = cpu_dual.to(dtype=torch.float32, device='cpu').contiguous()
gpu_dual_f32 = gpu_dual.to(dtype=torch.float32, device='cpu').contiguous()
cpu_inter_cpu = cpu_inter.to(device='cpu').contiguous()
gpu_inter_cpu = gpu_inter.to(device='cpu').contiguous()

def build_coord_map(coords_i64):
    # key: (x, y, z) -> row index
    m = {}
    dup = 0
    for i in range(coords_i64.shape[0]):
        k = tuple(int(v) for v in coords_i64[i].tolist())
        if k in m:
            dup += 1
        m[k] = i
    return m, dup

cpu_map, cpu_dup = build_coord_map(cpu_coords_i64)
gpu_map, gpu_dup = build_coord_map(gpu_coords_i64)

cpu_keys = set(cpu_map.keys())
gpu_keys = set(gpu_map.keys())
common_keys = sorted(cpu_keys & gpu_keys)
only_cpu = sorted(cpu_keys - gpu_keys)
only_gpu = sorted(gpu_keys - cpu_keys)

print('voxel geometry check')
print('  cpu voxel count:', len(cpu_keys), '(duplicate coords:', cpu_dup, ')')
print('  gpu voxel count:', len(gpu_keys), '(duplicate coords:', gpu_dup, ')')
print('  matched voxels :', len(common_keys))
print('  cpu-only voxels:', len(only_cpu))
print('  gpu-only voxels:', len(only_gpu))

if len(common_keys) > 0:
    cpu_idx = torch.tensor([cpu_map[k] for k in common_keys], dtype=torch.long)
    gpu_idx = torch.tensor([gpu_map[k] for k in common_keys], dtype=torch.long)

    cpu_dual_match = cpu_dual_f32[cpu_idx]
    gpu_dual_match = gpu_dual_f32[gpu_idx]

    cpu_finite = torch.isfinite(cpu_dual_match)
    gpu_finite = torch.isfinite(gpu_dual_match)
    both_finite = cpu_finite & gpu_finite

    cpu_nonfinite = int((~cpu_finite).sum().item())
    gpu_nonfinite = int((~gpu_finite).sum().item())
    either_nonfinite = int((~both_finite).sum().item())

    print('\ndual vertices finite-value check on matched voxels')
    print('  cpu non-finite elements :', cpu_nonfinite)
    print('  gpu non-finite elements :', gpu_nonfinite)
    print('  either-side non-finite  :', either_nonfinite)

    if int(both_finite.sum().item()) > 0:
        diff = cpu_dual_match - gpu_dual_match
        sq_err = diff.pow(2)
        sq_err_finite = sq_err[both_finite]

        mean_mse = float(sq_err_finite.mean().item())
        max_mse = float(sq_err_finite.max().item())
        rmse = float(torch.sqrt(sq_err_finite.mean()).item())

        print('\ndual vertices per-element error on matched voxels (finite elements only)')
        print('  mean MSE:', mean_mse)
        print('  max  MSE:', max_mse)
        print('  RMSE    :', rmse)
    else:
        print('\nNo finite dual-vertex elements to compare.')

    cpu_inter_match = cpu_inter_cpu[cpu_idx]
    gpu_inter_match = gpu_inter_cpu[gpu_idx]
    inter_same = (cpu_inter_match == gpu_inter_match).all(dim=1)
    inter_mismatch = int((~inter_same).sum().item())

    print('\nintersected consistency on matched voxels')
    print('  matched rows :', len(common_keys))
    print('  equal rows   :', int(inter_same.sum().item()))
    print('  mismatch rows:', inter_mismatch)
    print('  mismatch ratio:', inter_mismatch / len(common_keys))
else:
    print('\nNo matched voxels between CPU and GPU outputs; cannot compute dual/intersected comparisons.')

voxel geometry check
  cpu voxel count: 1441050 (duplicate coords: 0 )
  gpu voxel count: 1441050 (duplicate coords: 0 )
  matched voxels : 1441050
  cpu-only voxels: 0
  gpu-only voxels: 0

dual vertices finite-value check on matched voxels
  cpu non-finite elements : 0
  gpu non-finite elements : 0
  either-side non-finite  : 0

dual vertices per-element error on matched voxels (finite elements only)
  mean MSE: 1.7465549007056325e-09
  max  MSE: 1.5312387404264882e-05
  RMSE    : 4.1791805415414274e-05

intersected consistency on matched voxels
  matched rows : 1441050
  equal rows   : 1441050
  mismatch rows: 0
  mismatch ratio: 0.0


In [40]:
assert cpu_ok and gpu_ok, 'Run CPU/GPU cells first.'

cpu_coords, cpu_dual, cpu_inter = cpu_out
gpu_coords, gpu_dual, gpu_inter = gpu_out

cpu_coords_i64 = cpu_coords.to(dtype=torch.int64, device='cpu').contiguous()
gpu_coords_i64 = gpu_coords.to(dtype=torch.int64, device='cpu').contiguous()
cpu_dual_f32 = cpu_dual.to(dtype=torch.float32, device='cpu').contiguous()
gpu_dual_f32 = gpu_dual.to(dtype=torch.float32, device='cpu').contiguous()

cpu_map = {tuple(int(v) for v in cpu_coords_i64[i].tolist()): i for i in range(cpu_coords_i64.shape[0])}
gpu_map = {tuple(int(v) for v in gpu_coords_i64[i].tolist()): i for i in range(gpu_coords_i64.shape[0])}

common_keys = sorted(set(cpu_map.keys()) & set(gpu_map.keys()))
assert len(common_keys) > 0, 'No matched voxels.'

cpu_idx = torch.tensor([cpu_map[k] for k in common_keys], dtype=torch.long)
gpu_idx = torch.tensor([gpu_map[k] for k in common_keys], dtype=torch.long)

cpu_dual_match = cpu_dual_f32[cpu_idx]
gpu_dual_match = gpu_dual_f32[gpu_idx]

both_finite = torch.isfinite(cpu_dual_match) & torch.isfinite(gpu_dual_match)
finite_count = int(both_finite.sum().item())
all_count = int(both_finite.numel())

abs_err = (cpu_dual_match - gpu_dual_match).abs()
abs_err_finite = abs_err[both_finite]

eps = 1e-8
den = torch.maximum(cpu_dual_match.abs(), torch.full_like(cpu_dual_match, eps))
rel_err = abs_err / den
rel_err_finite = rel_err[both_finite]

print('matched voxels =', len(common_keys))
print('finite dual elements =', finite_count, '/', all_count)
print('finite ratio =', finite_count / all_count)

print('\nabsolute error summary')
print('  max abs err =', float(abs_err_finite.max().item()))
print('  mean abs err =', float(abs_err_finite.mean().item()))
print('  p95 abs err =', float(torch.quantile(abs_err_finite, 0.95).item()))
print('  p99 abs err =', float(torch.quantile(abs_err_finite, 0.99).item()))

print('\nrelative error summary (|cpu-gpu| / max(|cpu|, 1e-8))')
print('  max rel err =', float(rel_err_finite.max().item()))
print('  mean rel err =', float(rel_err_finite.mean().item()))
print('  p95 rel err =', float(torch.quantile(rel_err_finite, 0.95).item()))
print('  p99 rel err =', float(torch.quantile(rel_err_finite, 0.99).item()))

bad_1e3 = int((rel_err_finite > 1e-3).sum().item())
bad_1e2 = int((rel_err_finite > 1e-2).sum().item())
bad_1e1 = int((rel_err_finite > 1e-1).sum().item())
print('\nratio above thresholds')
print('  > 1e-3:', bad_1e3, '/', finite_count, '=', bad_1e3 / finite_count)
print('  > 1e-2:', bad_1e2, '/', finite_count, '=', bad_1e2 / finite_count)
print('  > 1e-1:', bad_1e1, '/', finite_count, '=', bad_1e1 / finite_count)

matched voxels = 1441050
finite dual elements = 4323150 / 4323150
finite ratio = 1.0

absolute error summary
  max abs err = 0.00391310453414917
  mean abs err = 2.9527145670726895e-06
  p95 abs err = 3.0994415283203125e-06
  p99 abs err = 1.1920928955078125e-05

relative error summary (|cpu-gpu| / max(|cpu|, 1e-8))
  max rel err = 187565.953125
  mean rel err = 0.5353795289993286
  p95 rel err = 9.835954188019969e-06
  p99 rel err = 0.00011704320786520839

ratio above thresholds
  > 1e-3: 9227 / 4323150 = 0.002134323352185328
  > 1e-2: 1165 / 4323150 = 0.0002694794305078473
  > 1e-1: 155 / 4323150 = 3.585348646241745e-05


In [41]:
assert cpu_ok and gpu_ok

cpu_coords, cpu_dual, _ = cpu_out
gpu_coords, gpu_dual, _ = gpu_out

cpu_coords_i64 = cpu_coords.to(dtype=torch.int64, device='cpu').contiguous()
gpu_coords_i64 = gpu_coords.to(dtype=torch.int64, device='cpu').contiguous()
cpu_dual_f32 = cpu_dual.to(dtype=torch.float32, device='cpu').contiguous()
gpu_dual_f32 = gpu_dual.to(dtype=torch.float32, device='cpu').contiguous()

cpu_map = {tuple(int(v) for v in cpu_coords_i64[i].tolist()): i for i in range(cpu_coords_i64.shape[0])}
gpu_map = {tuple(int(v) for v in gpu_coords_i64[i].tolist()): i for i in range(gpu_coords_i64.shape[0])}
common_keys = sorted(set(cpu_map.keys()) & set(gpu_map.keys()))

cpu_idx = torch.tensor([cpu_map[k] for k in common_keys], dtype=torch.long)
gpu_idx = torch.tensor([gpu_map[k] for k in common_keys], dtype=torch.long)

cpu_dual_match = cpu_dual_f32[cpu_idx]
gpu_dual_match = gpu_dual_f32[gpu_idx]

finite = torch.isfinite(cpu_dual_match) & torch.isfinite(gpu_dual_match)
abs_err = (cpu_dual_match - gpu_dual_match).abs()[finite]
scale = torch.maximum(cpu_dual_match.abs(), gpu_dual_match.abs())[finite]

# Symmetric relative error; better behaved than cpu-only denominator.
sym_rel = abs_err / torch.maximum(scale, torch.full_like(scale, 1e-8))

print('symmetric relative error (|cpu-gpu| / max(|cpu|,|gpu|,1e-8))')
print('  max =', float(sym_rel.max().item()))
print('  p95 =', float(torch.quantile(sym_rel, 0.95).item()))
print('  p99 =', float(torch.quantile(sym_rel, 0.99).item()))

for thr in [1e-3, 1e-2, 1e-1]:
    c = int((sym_rel > thr).sum().item())
    print(f'  > {thr}: {c} / {sym_rel.numel()} = {c / sym_rel.numel()}')

# Report on non-tiny magnitude entries to avoid near-zero blow-up.
mask_non_tiny = scale > 1e-3
if int(mask_non_tiny.sum().item()) > 0:
    rel_non_tiny = (abs_err[mask_non_tiny] / scale[mask_non_tiny])
    print('\nnon-tiny entries (max(|cpu|,|gpu|) > 1e-3)')
    print('  count =', int(rel_non_tiny.numel()))
    print('  max rel =', float(rel_non_tiny.max().item()))
    print('  p95 rel =', float(torch.quantile(rel_non_tiny, 0.95).item()))
    print('  p99 rel =', float(torch.quantile(rel_non_tiny, 0.99).item()))

symmetric relative error (|cpu-gpu| / max(|cpu|,|gpu|,1e-8))
  max = 1.0
  p95 = 9.835891432885546e-06
  p99 = 0.00011704320786520839
  > 0.001: 9224 / 4323150 = 0.002133629413737668
  > 0.01: 1155 / 4323150 = 0.0002671663023489816
  > 0.1: 149 / 4323150 = 3.446560956709807e-05

non-tiny entries (max(|cpu|,|gpu|) > 1e-3)
  count = 4319275
  max rel = 1.0
  p95 rel = 9.817938007472549e-06
  p99 rel = 0.00011523898865561932


## How To Read Failures

- If Cell 3 (JIT build) fails with unresolved symbols, native declarations/registrations are ahead of implementations.
- If CPU passes and GPU fails in Cells 7/8, Python dispatch is correct but GPU binding implementation/build wiring is incomplete.
- If both pass, the end-to-end API is likely usable for further numeric comparison and profiling.